In [1]:
import sys, os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
import sqlite3
import pandas as pd

from scoring.stats import compare_churn_cohort_engagement, wilson_confidence_interval, hypergeometric_test
from scoring.validation import evaluate_churn_detection

conn = sqlite3.connect("data/fan_analytics.db")
fans = pd.read_sql("SELECT * FROM fans", conn)
fans.head()

,fan_id,tenure_years,plan_tier,baseline_engagement,is_planted_churn,decline_start_week
0,1,18,standard,0.424548,0,NaN
1,2,2,premium,0.527589,0,NaN
2,3,9,standard,0.637180,0,NaN
3,4,5,club,0.517255,0,NaN
4,5,3,standard,0.681351,0,NaN


## Week-over-week trend via SQL window function

`LAG() OVER (PARTITION BY fan_id ORDER BY week)` looks back one row within
each fan's own week-ordered history to compute how much their score moved
since the prior week — a rolling comparison expressed entirely in SQL,
without pulling the data into pandas first.

In [3]:
trend_query = """
SELECT
    fan_id,
    week,
    engagement_score,
    engagement_score - LAG(engagement_score) OVER (PARTITION BY fan_id ORDER BY week) AS score_delta
FROM weekly_snapshots
ORDER BY fan_id, week
"""
trend = pd.read_sql(trend_query, conn)
trend.head(10)

,fan_id,week,engagement_score,score_delta
0,1,1,41.500000,NaN
1,1,2,42.966667,1.466667
2,1,3,42.400000,-0.566667
3,1,4,38.533333,-3.866667
4,1,5,42.033333,3.500000
5,1,6,46.300000,4.266667
6,1,7,45.966667,-0.333333
7,1,8,48.500000,2.533333
8,1,9,49.933333,1.433333
9,1,10,46.966667,-2.966667


## Engagement by plan tier, over time

A `JOIN` between the fact table (`weekly_snapshots`) and the dimension
table (`fans`) plus a `GROUP BY` on week and plan tier — the kind of
aggregation a BI report would run directly against this database.

In [4]:
tier_by_plan_query = """
SELECT
    w.week,
    f.plan_tier,
    AVG(w.engagement_score) AS avg_engagement_score,
    COUNT(*) AS n_fans
FROM weekly_snapshots w
JOIN fans f ON w.fan_id = f.fan_id
GROUP BY w.week, f.plan_tier
ORDER BY w.week, f.plan_tier
"""
by_plan_tier = pd.read_sql(tier_by_plan_query, conn)
by_plan_tier.tail(10)

,week,plan_tier,avg_engagement_score,n_fans
44,15,standard,52.391149,177
45,16,club,52.269524,35
46,16,premium,44.654545,88
47,16,standard,52.491337,177
48,17,club,52.222857,35
49,17,premium,44.551136,88
50,17,standard,52.551977,177
51,18,club,52.176190,35
52,18,premium,44.478409,88
53,18,standard,52.597363,177


## Reconstructing the at-risk list in pure SQL

Notebook 03 produced the final week's at-risk list using pandas. Here's
the same result arrived at a different way — a CTE finds the latest week,
then filters `weekly_snapshots` down to flagged fans in that week. This is
a cross-check: if these two independently-computed lists ever disagreed,
that would mean a bug somewhere in the pipeline.

In [5]:
at_risk_query = """
WITH final_week AS (
    SELECT MAX(week) AS max_week FROM weekly_snapshots
)
SELECT
    w.fan_id,
    w.engagement_score,
    w.tier
FROM weekly_snapshots w, final_week
WHERE w.week = final_week.max_week AND w.at_risk = 1
ORDER BY w.engagement_score ASC
"""
at_risk_sql = pd.read_sql(at_risk_query, conn)
at_risk_sql

,fan_id,engagement_score,tier
0,165,1.433333,Dormant
1,232,3.966667,Dormant
2,142,4.566667,Dormant
3,67,5.266667,Dormant
4,238,8.366667,Dormant
5,96,10.600000,Dormant
6,227,11.566667,Dormant
7,216,13.733333,Dormant
8,244,13.933333,Dormant
9,34,14.733333,Dormant


## Statistical test 1 — is the planted-churn cohort's decline real?

A Mann-Whitney U test compares the final week's engagement scores for the
planted-churn cohort against everyone else. Mann-Whitney is used instead
of a t-test because `engagement_score` is a population-relative percentile
rank by construction, not a normally-distributed measurement — the test
should not assume a distribution shape the data doesn't have. The
alternative hypothesis is one-sided (`"less"`): planted-churn fans are
expected to score lower, not just "different."

In [6]:
final_week_num = int(pd.read_sql("SELECT MAX(week) AS w FROM weekly_snapshots", conn)["w"].iloc[0])
final_scores = pd.read_sql(
    f"SELECT fan_id, engagement_score FROM weekly_snapshots WHERE week = {final_week_num}", conn
)

mw_result = compare_churn_cohort_engagement(final_scores, fans)
print(f"Mann-Whitney U p-value: {mw_result['p_value']:.6f}")
print(f"Planted-churn median score: {mw_result['planted_median']:.2f}")
print(f"Everyone-else median score: {mw_result['rest_median']:.2f}")

Mann-Whitney U p-value: 0.000000
Planted-churn median score: 5.60
Everyone-else median score: 54.63


## Statistical test 2 — how confident are we in precision and recall?

Precision and recall are point estimates from a small sample (a handful
of flagged fans out of 300). A Wilson score confidence interval gives a
plausible range for each, rather than reporting a single number as if it
were exact. Wilson is used instead of a naive normal-approximation
interval because it stays well-behaved for small counts close to 0 or 1.

In [7]:
at_risk_final = pd.read_sql(
    f"SELECT fan_id, {final_week_num} AS week, at_risk FROM weekly_snapshots WHERE week = {final_week_num}", conn
)
validation = evaluate_churn_detection(at_risk_final, fans, week=final_week_num)

precision_ci = wilson_confidence_interval(
    validation["true_positives"], validation["true_positives"] + validation["false_positives"]
)
recall_ci = wilson_confidence_interval(
    validation["true_positives"], validation["true_positives"] + validation["false_negatives"]
)

print(f"Precision: {validation['precision']:.2f}  95% CI: ({precision_ci['lower']:.2f}, {precision_ci['upper']:.2f})")
print(f"Recall: {validation['recall']:.2f}  95% CI: ({recall_ci['lower']:.2f}, {recall_ci['upper']:.2f})")

Precision: 0.62  95% CI: (0.36, 0.82)
Recall: 0.32  95% CI: (0.17, 0.52)


## Statistical test 3 — could this many true positives happen by chance?

If you flagged fans at random instead of using the churn rule, how many
true positives would you expect to get lucky into? A hypergeometric test
answers this exactly: it models drawing a fixed number of fans without
replacement from a finite population with a known number of true
churners in it — which is exactly what's happening here — rather than
approximating with a binomial distribution, which assumes draws don't
change the population (they do, since a fan can't be flagged twice).

In [8]:
n_planted = int(fans["is_planted_churn"].sum())
n_flagged = validation["true_positives"] + validation["false_positives"]

hg_result = hypergeometric_test(
    population_size=len(fans),
    n_true_churners=n_planted,
    n_flagged=n_flagged,
    n_true_positives=validation["true_positives"],
)
print(f"P(>= {validation['true_positives']} true positives by chance): {hg_result['p_value']:.6f}")
print(f"Expected true positives by chance: {hg_result['expected_true_positives_by_chance']:.2f}")

P(>= 8 true positives by chance): 0.000001
Expected true positives by chance: 1.08
